In [ ]:
import requests
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath(".."))
from logs.logger import logger
import time

/Users/sburtsev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
log = logger()

IGDB

Начнем с того, что попробуем взять igdb_id, igdb_secret, потом получить от них токен и попробовать с ним поработать для первичного анализа инди игр

Документация IGDB API- https://api-docs.igdb.com/ 



In [3]:
igdb_id = os.getenv('client_id')
igdb_secret = os.getenv('client_secret')


In [4]:
request = requests.post(url = f'https://id.twitch.tv/oauth2/token?client_id={igdb_id}&client_secret={igdb_secret}&grant_type=client_credentials')
request #получили токен, добавили в .env как access_token

<Response [200]>

In [6]:
if request.status_code != 200:
    log.error('Ошибка получения токена')

In [7]:
access_token = os.getenv('access_token')

Имея датасет из примерно примерно 27к игр жанра Инди (имея их steam_id), получим их же id на igdb, чтобы далее с помощью других эндпоинтов брать информацию

Для начала очистим датасет от (на данный момент) ненужных столбцов и попробуем получить 500 строк, если все будет в порядке, напишем цикл, который возьмет остальные нужные (так как в igdb можно брать максимум 500 строк за раз)

In [8]:
df_parsed = pd.read_excel('../../data/df_parsed.xlsx')
df_parsed.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26853 entries, 0 to 26852
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   steam_id                    26853 non-null  int64  
 1   game_name                   26412 non-null  object 
 2   game_description_snippet    25870 non-null  object 
 3   game_price                  20284 non-null  object 
 4   found_game_price            25433 non-null  float64
 5   all_language_reviews_type   25042 non-null  object 
 6   all_language_reviews_count  25042 non-null  float64
 7   has_russian_reviews         25433 non-null  float64
 8   all_russian_reviews_type    1559 non-null   object 
 9   all_russian_reviews_count   1559 non-null   float64
 10  steam_app_url               25860 non-null  object 
 11  support_ru_region           25860 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 2.5+ MB


In [9]:
df_parsed = df_parsed.drop(columns= ['game_description_snippet', 'game_price', 'found_game_price', 'all_language_reviews_type',
                             'all_language_reviews_count', 'has_russian_reviews', 'all_russian_reviews_type', 'all_russian_reviews_count', 
                             'steam_app_url', 'support_ru_region' ])
df_parsed = df_parsed.rename(columns ={'steam_id': 'steam_app_id', 'game_name': 'name'})
df_parsed

,steam_app_id,name
0,1313940,My Holiday 2
1,1902870,Vertical Kingdom
2,1071940,Streamer's Life
3,1706570,Chupacabras: Night Hunt
4,1085150,Golf Defied
...,...,...
26848,1547900,Active Mummy
26849,2238360,You Have No Time
26850,1490030,Cyber Dodge
26851,1855190,Vektor Z


In [10]:
ids = df_parsed['steam_app_id'].astype(str).tolist()[:500]
uids = '","'.join(ids)
uids = '"' + uids + '"'


data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data)
res = request_games_steam.json()
res

map_df = pd.DataFrame(res)
map_df['uid'] = map_df['uid'].astype(int)

df_parsed = df_parsed.rename(columns ={'steam_app_id': 'uid'})
df_parsed['uid'] = df_parsed['uid'].astype(int)
out = df_parsed.merge(map_df, on= 'uid', how= 'left')

if request_games_steam.status_code != 200:
    log.error('Ошибка получения external_games')

In [11]:
out

,uid,name,id,game
0,1313940,My Holiday 2,2035995.0,150866.0
1,1902870,Vertical Kingdom,2447084.0,202602.0
2,1071940,Streamer's Life,1720801.0,124447.0
3,1706570,Chupacabras: Night Hunt,2080403.0,163904.0
4,1085150,Golf Defied,1722413.0,118634.0
...,...,...,...,...
26849,1547900,Active Mummy,NaN,NaN
26850,2238360,You Have No Time,NaN,NaN
26851,1490030,Cyber Dodge,NaN,NaN
26852,1855190,Vektor Z,NaN,NaN


Все классно достается, запустим цикл, который соберет игры пачками по 500 за раз

In [12]:
all = []

for i in range(0, len(df_parsed), 500):
    ids = df_parsed['uid'].iloc[i:i+500].astype(str).tolist()
    uids = '"' + '","'.join(ids) + '"'

    data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
    request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data, timeout = 60)
    
    if request_games_steam.status_code == 200:
        all += request_games_steam.json()
    else:
        log.error(f'пачка игра {i//500+ 1} ошибка со статусом {request_games_steam.status_code}')

    time.sleep(0.5)

map_df = pd.DataFrame(all)
map_df['uid'] = map_df['uid'].astype(int)

out = df_parsed.merge(map_df, on='uid', how ='left')
out

,uid,name,id,game
0,1313940,My Holiday 2,2035995.0,150866.0
1,1902870,Vertical Kingdom,2447084.0,202602.0
2,1071940,Streamer's Life,1720801.0,124447.0
3,1706570,Chupacabras: Night Hunt,2080403.0,163904.0
4,1085150,Golf Defied,1722413.0,118634.0
...,...,...,...,...
26849,1547900,Active Mummy,2041722.0,156747.0
26850,2238360,You Have No Time,2690119.0,244854.0
26851,1490030,Cyber Dodge,2031292.0,159989.0
26852,1855190,Vektor Z,2221496.0,186672.0


Что мы получили? 
1) uid - id игры в стиме
2) name - название игры
3) game - id игры в igdb

In [13]:
out = out.dropna() #удалим пропуски, так как далее не сможем с ними ничего сделать в других эндпоинтах (пропуски в game)
print(out.isna().sum())
out.info()

uid     0
name    0
id      0
game    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 25702 entries, 0 to 26853
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   uid     25702 non-null  int64  
 1   name    25702 non-null  object 
 2   id      25702 non-null  float64
 3   game    25702 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1004.0+ KB


In [14]:
out['game'] = out['game'].astype(int)
out.to_csv('../../data/igdb_steam_games_final.csv', index=False) 
log.info('Датасет успешно сохранен в файл igdb_steam_games_final.csv')
out['game']


/var/folders/nh/1j69s9x56_x30mtmw81z3p9r0000gn/T/ipykernel_96824/742260524.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['game'] = out['game'].astype(int)
2026-04-09 16:17:02,010 - 742260524.py - INFO - Датасет успешно сохранен в файл igdb_steam_games_final.csv


0        150866
1        202602
2        124447
3        163904
4        118634
          ...  
26849    156747
26850    244854
26851    159989
26852    186672
26853    186861
Name: game, Length: 25702, dtype: int64

Далее, по эндпоинту games, достанем кое-какие подробности по нашим играм, и сохраним датасет 

In [15]:
prob = out.head(500).copy()
prob
ids = prob['game'].astype(str).tolist()
ids_str = ','.join(ids)

data1 = f'''
fields id, name, rating, rating_count, total_rating, total_rating_count, first_release_date; where id = ({ids_str}); limit 500;
'''
request_prob = requests.post('https://api.igdb.com/v4/games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1)
if request_prob.status_code != 200:
    log.error('Ошибка получения сэмпла на 500 игр')

res =request_prob.json()
res

[{'id': 9012,
  'first_release_date': 1386115200,
  'name': 'Robot Rescue Revolution'},
 {'id': 9865,
  'first_release_date': 1399334400,
  'name': 'Sportsfriends',
  'rating': 62.15205072670145,
  'rating_count': 21,
  'total_rating': 71.79031107763645,
  'total_rating_count': 28},
 {'id': 10336, 'first_release_date': 1640822400, 'name': 'Frogs vs. Storks'},
 {'id': 11644, 'first_release_date': 1439164800, 'name': 'One Final Breath'},
 {'id': 13359,
  'first_release_date': 1445299200,
  'name': 'Pulse',
  'rating': 40.0,
  'rating_count': 1,
  'total_rating': 50.83333333333333,
  'total_rating_count': 4},
 {'id': 13739, 'first_release_date': 1482192000, 'name': 'The Mine'},
 {'id': 16266,
  'first_release_date': 1132617600,
  'name': 'Mutant Storm: Reloaded',
  'rating': 60.0,
  'rating_count': 1,
  'total_rating': 72.5,
  'total_rating_count': 3},
 {'id': 16548,
  'first_release_date': 1380585600,
  'name': 'Rain Blood Chronicles: Mirage',
  'rating': 60.0,
  'rating_count': 3,
  'to

In [16]:
df_games_igdb = pd.DataFrame(res)
df_games_igdb.isna().sum()

id                      0
first_release_date      8
name                    0
rating                399
rating_count          399
total_rating          393
total_rating_count    393
dtype: int64

Получается неплохо, берем оставшиеся игры 

In [35]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields id, name, genres.name, rating, rating_count, total_rating, total_rating_count, first_release_date; where id = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout = 60)
    if request.status_code == 200:
        all += request.json()
    else:
        log.error(f'пачка игра {i//500+ 1} ошибка со статусом {request.status_code}')
    time.sleep(0.5)

df_games = pd.DataFrame(all)
df_games

,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,1.386115e+09,"[{'id': 9, 'name': 'Puzzle'}, {'id': 32, 'name...",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,1.399334e+09,"[{'id': 4, 'name': 'Fighting'}, {'id': 14, 'na...",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,1.640822e+09,"[{'id': 9, 'name': 'Puzzle'}, {'id': 15, 'name...",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,1.439165e+09,"[{'id': 13, 'name': 'Simulator'}, {'id': 31, '...",One Final Breath,NaN,NaN,NaN,NaN
4,13359,1.445299e+09,"[{'id': 31, 'name': 'Adventure'}, {'id': 32, '...",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,1.736381e+09,"[{'id': 12, 'name': 'Role-playing (RPG)'}, {'i...",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,1.734048e+09,"[{'id': 9, 'name': 'Puzzle'}, {'id': 13, 'name...",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,1.741392e+09,"[{'id': 9, 'name': 'Puzzle'}, {'id': 31, 'name...",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,1.568160e+09,"[{'id': 32, 'name': 'Indie'}]",Ain Dodo,NaN,NaN,NaN,NaN


In [36]:
df_games.isna().sum()

id                        0
first_release_date      334
genres                  166
name                      0
rating                15578
rating_count          15578
total_rating          15144
total_rating_count    15144
dtype: int64

Также, сразу немного преобразуем жанры

In [37]:
df_games['genres'] = df_games['genres'].astype(str)
df_games['genres'] = df_games['genres'].str.findall(r"'name': '([^']+)'").str.join(', ')
df_games

,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,1.386115e+09,"Puzzle, Indie",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,1.399334e+09,"Fighting, Sport, Indie, Arcade",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,1.640822e+09,"Puzzle, Strategy, Indie, Card & Board Game",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,1.439165e+09,"Simulator, Adventure, Indie",One Final Breath,NaN,NaN,NaN,NaN
4,13359,1.445299e+09,"Adventure, Indie",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,1.736381e+09,"Role-playing (RPG), Indie",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,1.734048e+09,"Puzzle, Simulator, Indie",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,1.741392e+09,"Puzzle, Adventure, Indie",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,1.568160e+09,Indie,Ain Dodo,NaN,NaN,NaN,NaN


Пока что особо проблем с данными нет, кроме рейтинга, пропущенных немало, есть несколько столбцов, которых нужно немного преобразовать (можно сделать, когда соберем полный датасет), но в общем и целом, с такими данными уже можно работать

Сохраним пока датасет

In [38]:
df_games.to_csv('../../data/df_games_final.csv', index= False)
log.info('Датасет успешно сохранен в файл df_games_final.csv')

2026-04-09 17:44:18,746 - 2840804000.py - INFO - Датасет успешно сохранен в файл df_games_final.csv


Далее, посмотрим эндпоинт про популярность и попробуем взять наши игры 

In [39]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields game_id, external_popularity_source.name, value; where game_id = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/popularity_primitives', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout= 90)
    if request.status_code == 200:
        all += request.json()
    else:
        log.error(f'пачка игра {i//500+ 1} ошибка со статусом {request.status_code}')
    time.sleep(1)
    
df_popular = pd.DataFrame(all)
df_popular

,id,game_id,value,external_popularity_source
0,98306,9012,0.000003,"{'id': 121, 'name': 'IGDB'}"
1,3372,9865,0.000028,"{'id': 121, 'name': 'IGDB'}"
2,98958,9865,0.000016,"{'id': 121, 'name': 'IGDB'}"
3,4845023,9865,0.000002,"{'id': 121, 'name': 'IGDB'}"
4,4751475,9865,0.000004,"{'id': 121, 'name': 'IGDB'}"
...,...,...,...,...
23729,4787973,20102,0.000013,"{'id': 121, 'name': 'IGDB'}"
23730,4801559,247501,0.000004,"{'id': 121, 'name': 'IGDB'}"
23731,4801842,250540,0.000004,"{'id': 121, 'name': 'IGDB'}"
23732,4802266,255582,0.000004,"{'id': 121, 'name': 'IGDB'}"


In [40]:
#сразу поправим столбец и посмотрим по каким источникам у нас популярность считается
df_popular['external_popularity_source'] =df_popular['external_popularity_source'].apply(lambda x: x['name'])
df_popular['external_popularity_source'].unique()

array(['IGDB', 'Twitch', 'Steam'], dtype=object)

Видим, что популярность считается по игдб, стиму и твичу, сохраним пока это все в отдельный датасет

In [41]:
df_popular.to_csv('../../data/df_popular_final.csv', index= False)
log.info('Датасет успешно сохранен в файл df_popular_final.csv')

2026-04-09 17:46:10,976 - 3622898071.py - INFO - Датасет успешно сохранен в файл df_popular_final.csv


Далее, возьмем еще один эндпоинт о присутствии игры на каких-либо площадках по типу Steam, Epic и т.д., чтобы на этапе анализа (eda) оценить возможность связи с популярностью, или ценой к примеру

In [42]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields game, type.type; where game = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/websites', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout= 90)
    if request.status_code == 200:
        all += request.json()
    else:
        log.error(f'пачка игра {i//500+ 1} ошибка со статусом {request.status_code}')
    time.sleep(1)

df_web = pd.DataFrame(all)
df_web

,id,game,type
0,392535,129982,"{'id': 6, 'type': 'Twitch'}"
1,743372,201352,"{'id': 16, 'type': 'Epic'}"
2,469527,171379,"{'id': 6, 'type': 'Twitch'}"
3,672929,291422,"{'id': 5, 'type': 'Twitter'}"
4,672926,291422,"{'id': 1, 'type': 'Official Website'}"
...,...,...,...
25995,248376,114968,"{'id': 15, 'type': 'Itch'}"
25996,248377,114968,"{'id': 5, 'type': 'Twitter'}"
25997,252983,119339,"{'id': 18, 'type': 'Discord'}"
25998,252981,119339,"{'id': 5, 'type': 'Twitter'}"


In [43]:
df_web['type'] =df_web['type'].apply(lambda x: x['type'])
df_web.isna().sum()

id      0
game    0
type    0
dtype: int64

In [44]:
df_web.to_csv('../../data/df_web_final.csv', index=False)
log.info('Датасет успешно сохранен в файл df_web_final.csv')


2026-04-09 17:48:06,510 - 3564491737.py - INFO - Датасет успешно сохранен в файл df_web_final.csv
